In [1]:
import sys # 파이썬 시스템 정보를 다루는 라이브러리를 불러옵니다.
# 현재 노트북이 실행 중인 파이썬 경로를 찾아 그 경로에 직접 minsearch를 설치합니다.
!{sys.executable} -m pip install minsearch 


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: E:\IT_SPACES\AI\ZoomCamp\LLM\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [3]:
documents[2]

{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
 'section': 'General course-related questions',
 'question': 'Course - Can I still join the course after the start date?',
 'course': 'data-engineering-zoomcamp'}

In [4]:
import minsearch

index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [5]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_llm_root = next(
    (p for p in [_cwd, *_cwd.parents] if (p / "openai_env.py").is_file()),
    None,
)
if _llm_root is None:
    raise RuntimeError(
        "LLM 폴더(openai_env.py)를 찾지 못했습니다. "
        "Jupyter 작업 디렉터리가 ZoomCamp\\LLM\\02\\Vector_Search 근처인지 확인하세요."
    )
sys.path.insert(0, str(_llm_root))

from openai_env import get_openai_client

# API 키: ZoomCamp\\LLM\\.env 의 OPENAI_API_KEY=... (예시는 .env.example 참고, .env 는 Git 커밋 금지)
client = get_openai_client()


In [6]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [7]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [8]:
def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [9]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)

    return answer

In [10]:
rag('how do I run kafka?')

'To run Kafka, if you\'re working with Java and need to run a producer, you can use the following command in the project directory:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```\n\nIf you\'re using Python and encounter a "Module \'kafka\' not found" error, you should create a virtual environment, activate it, and install the necessary packages from `requirements.txt`. Here\'s how you can do it:\n\n1. Create and activate a virtual environment:\n\n   ```bash\n   python -m venv env\n   source env/bin/activate\n   ```\n\n   For Windows, use:\n\n   ```bash\n   env\\Scripts\\activate\n   ```\n\n2. Install the required packages:\n\n   ```bash\n   pip install -r ../requirements.txt\n   ```\n\n3. Make sure all Docker images are up and running before executing the Python file in the virtual environment. Use `deactivate` to exit the virtual environment when you\'re done.'

In [11]:
rag('the course has already started, can I still enroll?')

"Yes, you can still enroll in the course after it has started. You are eligible to submit the homeworks regardless of registration. However, keep in mind that there are deadlines for submitting the final projects, so it's important not to leave everything until the last minute."